<a href="https://colab.research.google.com/github/MahderAkalu/Comparative-Evaluation-of-Privacy-Preserving-Techniques-in-Deep-Federated-Learning/blob/master/CIFAR_100423.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================================
# FULL BENCHMARK NOTEBOOK
# Comparative Evaluation of Privacy-Preserving Federated Learning
# Dataset : CIFAR-10
# Model   : Small CNN
# Framework: Flower + PyTorch + Opacus
#
# Methods:
#   R0  = FedAvg (baseline)
#   R1a = DP-FL  (sigma=0.5)
#   R1b = DP-FL  (sigma=1.0)
#   R2  = Secure Aggregation simulation
#
# Seeds: [42, 123, 7]
#
# Metrics:
#   Utility  : Accuracy / F1 / ROC-AUC
#   Privacy  : epsilon (RDP accounting)
#   Empirical: MIA AUC / DLG proxy score
#   Systems  : Comm GB / Wall-clock hours
#




# 1. INSTALL


!pip uninstall -y flwr flwr-datasets ray grpcio protobuf numpy pandas -q

!pip install -q \
numpy==1.26.4 \
pandas==2.2.2 \
scikit-learn==1.5.0 \
"flwr[simulation]==1.10.0" \
flwr-datasets \
"ray==2.34.0" \
grpcio \
protobuf \
opacus \
torch torchvision tqdm \
matplotlib seaborn


# 2. IMPORTS

import os
import time
import random
import math
import json
import copy
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Tuple, Union

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision.models as models
from torchvision.transforms import (
    Compose, ToTensor, Normalize,
    RandomCrop, RandomHorizontalFlip,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)

import flwr as fl
from flwr.common import (
    Parameters,
    FitRes,
    Scalar,
    ndarrays_to_parameters,
    parameters_to_ndarrays,
)
from flwr.server.client_proxy import ClientProxy
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import DirichletPartitioner
from datasets import load_dataset

from opacus import PrivacyEngine
from opacus.accountants import RDPAccountant

import ray
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)



# 3. GLOBAL CONFIG

SEEDS        = [42, 123, 7]

NUM_CLIENTS  = 10
NUM_ROUNDS   = 10
FRAC_FIT     = 0.5
LOCAL_EPOCHS = 3
BATCH_SIZE   = 64

ALPHA        = 0.5         # Dirichlet heterogeneity
CLIP_NORM    = 1.0
DELTA        = 1e-5

TARGET_ACC   = 0.75


USE_RESNET   = False

RESULTS_CSV  = "benchmark_results.csv"

DP_SIGMAS    = [0.5, 1.0]



# 4. EXPERIMENT CONFIG DATACLASS

@dataclass
class ExperimentConfig:
    method:     str
    dp:         bool  = False
    sigma:      float = 0.0
    secureagg:  bool  = False


EXPERIMENTS = [
    ExperimentConfig("R0_FedAvg"),
    ExperimentConfig("R1a_DP_sigma0.5", dp=True,  sigma=0.5),
    ExperimentConfig("R1b_DP_sigma1.0", dp=True,  sigma=1.0),
    ExperimentConfig("R2_SecureAgg",    secureagg=True),
]



# 5. REPRODUCIBILITY

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



# 6. MODEL

class SmallCNN(nn.Module):
            
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 16×16
            nn.Dropout2d(0.25),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 8×8
            nn.Dropout2d(0.25),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


class ResNet18(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = models.resnet18(weights=None)
        self.model.fc = nn.Linear(512, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def Net() -> nn.Module:
    return ResNet18() if USE_RESNET else SmallCNN()


def get_parameters(net: nn.Module) -> List[np.ndarray]:
    return [v.cpu().numpy() for _, v in net.state_dict().items()]


def set_parameters(net: nn.Module, params: List[np.ndarray]) -> None:
    state = {k: torch.tensor(v) for k, v in zip(net.state_dict().keys(), params)}
    net.load_state_dict(state, strict=True)



# 7. DATA

train_transform = Compose([
    RandomCrop(32, padding=4),
    RandomHorizontalFlip(),
    ToTensor(),
    Normalize((0.4914, 0.4822, 0.4465),
              (0.2023, 0.1994, 0.2010)),
])

test_transform = Compose([
    ToTensor(),
    Normalize((0.4914, 0.4822, 0.4465),
              (0.2023, 0.1994, 0.2010)),
])

_dataset_cache: Dict[int, Any] = {}


def get_dataset(seed: int):
    if seed in _dataset_cache:
        return _dataset_cache[seed]

    fds = FederatedDataset(
        dataset="cifar10",
        partitioners={
            "train": DirichletPartitioner(
                num_partitions=NUM_CLIENTS,
                alpha=ALPHA,
                partition_by="label",
                seed=seed,
            )
        },
    )

    testset = load_dataset("cifar10", split="test")

    def apply_test(batch):
        batch["img"] = [test_transform(x) for x in batch["img"]]
        return batch

    testset.set_transform(apply_test)

    # Split test set: 80% utility eval / 20% MIA non-members
    n_test      = len(testset)
    split_idx   = int(0.8 * n_test)
    eval_idx    = list(range(split_idx))
    mia_idx     = list(range(split_idx, n_test))

    eval_loader = DataLoader(
        Subset(testset, eval_idx),
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True,
    )
    mia_test_loader = DataLoader(
        Subset(testset, mia_idx),
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True,
    )

    _dataset_cache[seed] = (fds, eval_loader, mia_test_loader)
    return _dataset_cache[seed]


def load_partition(cid: int, fds: FederatedDataset, seed: int):

    part  = fds.partitioners["train"].load_partition(cid)
    split = part.train_test_split(test_size=0.1, seed=seed)

    trainset = split["train"]
    valset   = split["test"]

    def apply_train(batch):
        batch["img"] = [train_transform(x) for x in batch["img"]]
        return batch

    def apply_val(batch):
        batch["img"] = [test_transform(x) for x in batch["img"]]
        return batch

    trainset.set_transform(apply_train)
    valset.set_transform(apply_val)

    trainloader = DataLoader(
        trainset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )
    valloader = DataLoader(
        valset,
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True,
    )

    return trainloader, valloader


# 8. TRAINING

def train(
    net: nn.Module,
    loader: DataLoader,
    epochs: int,
    dp: bool    = False,
    sigma: float = 0.0,
    clip: float  = 1.0,
):
    net.to(DEVICE)
    net.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        net.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=5e-4,
    )

    if dp:
        privacy_engine = PrivacyEngine()
        net, optimizer, loader = privacy_engine.make_private(
            module=net,
            optimizer=optimizer,
            data_loader=loader,
            noise_multiplier=sigma,
            max_grad_norm=clip,
        )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    actual_steps = 0
    for _ in range(epochs):
        for batch in loader:
            x = batch["img"].to(DEVICE)
            y = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            loss = criterion(net(x), y)
            loss.backward()
            optimizer.step()
            actual_steps += 1

        scheduler.step()

    return net, len(loader.dataset), actual_steps



# 9. EVALUATION

def evaluate_model(net: nn.Module, loader: DataLoader):

    net.to(DEVICE)
    net.eval()

    preds, probs, labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            x = batch["img"].to(DEVICE)
            y = batch["label"].cpu().numpy()

            p = F.softmax(net(x), dim=1).cpu().numpy()
            preds.extend(np.argmax(p, axis=1))
            probs.extend(p)
            labels.extend(y)

    preds  = np.array(preds)
    probs  = np.array(probs)
    labels = np.array(labels)

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(labels, probs, multi_class="ovr")

    return acc, f1, auc


# 10. COMMUNICATION SIZE

def model_size_gb(net: nn.Module) -> float:
    total = sum(p.numel() * 4 for p in net.parameters())
    return total / (1024 ** 3)



# 11. PRIVACY ACCOUNTING

def estimate_epsilon(
    sigma: float,
    sample_rate: float,
    steps: int,
    delta: float = DELTA,
) -> Optional[float]:
  
    if sigma <= 0:
        return None

    accountant = RDPAccountant()
    for _ in range(steps):
        accountant.step(noise_multiplier=sigma, sample_rate=sample_rate)

    return accountant.get_epsilon(delta=delta)



# 12. MEMBERSHIP INFERENCE ATTACK (confidence-based)

def run_mia(
    net: nn.Module,
    member_loader: DataLoader,
    nonmember_loader: DataLoader,
) -> float:
   
    net.eval()

    def confidence_scores(loader: DataLoader) -> List[float]:
        scores = []
        with torch.no_grad():
            for batch in loader:
                x   = batch["img"].to(DEVICE)
                out = F.softmax(net(x), dim=1)
                scores.extend(out.max(dim=1)[0].cpu().tolist())
        return scores

    train_scores = confidence_scores(member_loader)
    test_scores  = confidence_scores(nonmember_loader)

    y_true  = np.array([1] * len(train_scores) + [0] * len(test_scores))
    y_score = np.array(train_scores + test_scores)

    return roc_auc_score(y_true, y_score)



# 13. DLG PROXY SCORE

def run_dlg_proxy(net: nn.Module) -> float:
    total, count = 0.0, 0
    for p in net.parameters():
        g      = p.detach().cpu().numpy().flatten()
        total += float(np.mean(np.abs(g)))
        count += 1
    return total / max(count, 1)



# 14. SAVE-PARAMS STRATEGY

class SaveParamsStrategy(fl.server.strategy.FedAvg):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.final_parameters: Optional[Parameters] = None

    def aggregate_fit(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, FitRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[Parameters], Dict[str, Scalar]]:

        aggregated_params, metrics = super().aggregate_fit(
            server_round, results, failures
        )

        if aggregated_params is not None:
            self.final_parameters = aggregated_params

        return aggregated_params, metrics



# 15. FLOWER CLIENT

class FlowerClient(fl.client.NumPyClient):

    def __init__(
        self,
        cid: int,
        trainloader: DataLoader,
        valloader: DataLoader,
        dp: bool,
        sigma: float,
    ):
        self.cid         = cid
        self.trainloader = trainloader
        self.valloader   = valloader
        self.net         = Net()
        self.dp          = dp
        self.sigma       = sigma

    def get_parameters(self, config):
        return get_parameters(self.net)

    def fit(self, params, config):
        set_parameters(self.net, params)

        trained_net, n, steps = train(
            self.net,
            self.trainloader,
            LOCAL_EPOCHS,
            self.dp,
            self.sigma,
            CLIP_NORM,
        )

        
        self.net = trained_net

        return get_parameters(self.net), n, {"steps": steps}

    def evaluate(self, params, config):
        set_parameters(self.net, params)
        acc, f1, auc = evaluate_model(self.net, self.valloader)
        return float(1 - acc), len(self.valloader.dataset), {"accuracy": acc}



# 16. RUN ONE EXPERIMENT

def run_experiment(seed: int, cfg: ExperimentConfig) -> Dict[str, Any]:

    set_seed(seed)

    fds, eval_loader, mia_test_loader = get_dataset(seed)

    
    def client_fn(cid):
        trainloader, valloader = load_partition(int(cid), fds, seed)
        return FlowerClient(int(cid), trainloader, valloader, cfg.dp, cfg.sigma)

    init_net  = Net()
    strategy  = SaveParamsStrategy(
        fraction_fit          = FRAC_FIT,
        fraction_evaluate     = 0.0,
        min_fit_clients       = max(2, int(NUM_CLIENTS * FRAC_FIT)),
        min_available_clients = NUM_CLIENTS,
        initial_parameters    = ndarrays_to_parameters(
            get_parameters(init_net)
        ),
    )


    start = time.time()

    fl.simulation.start_simulation(
        client_fn   = client_fn,
        num_clients = NUM_CLIENTS,
        config      = fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy    = strategy,
        ray_init_args={
            "ignore_reinit_error": True,
            "log_to_driver": False,
        },
    )

    runtime_hours = (time.time() - start) / 3600

    final_net = Net()
    if strategy.final_parameters is not None:
        final_params = parameters_to_ndarrays(strategy.final_parameters)
        set_parameters(final_net, final_params)
        print(f"  ✓ Loaded trained weights from SaveParamsStrategy.")
    else:
        print(
            f"[WARN] seed={seed} method={cfg.method}: "
            "SaveParamsStrategy captured no parameters — "
            "accuracy will be ~chance. Check that at least one round completed."
        )

 
    acc, f1, auc = evaluate_model(final_net, eval_loader)

    
    base_gb = model_size_gb(final_net)
    comm_gb = base_gb * NUM_CLIENTS * FRAC_FIT * NUM_ROUNDS * 2  
    if cfg.secureagg:
        comm_gb *= 1.35  

    
    eps = None
    if cfg.dp:
       
        partition_size    = 50_000 // NUM_CLIENTS
        batches_per_epoch = math.ceil(partition_size / BATCH_SIZE)
        total_steps = (
            NUM_ROUNDS
            * max(2, int(NUM_CLIENTS * FRAC_FIT))
            * LOCAL_EPOCHS
            * batches_per_epoch
        )
        sample_rate = BATCH_SIZE / partition_size
        eps = estimate_epsilon(cfg.sigma, sample_rate, total_steps)

    
    member_loader, _ = load_partition(0, fds, seed)

    mia_auc = run_mia(final_net, member_loader, mia_test_loader)
    dlg     = run_dlg_proxy(final_net)

    return {
        "Seed":        seed,
        "Method":      cfg.method,
        "Accuracy":    round(acc,  4),
        "F1":          round(f1,   4),
        "AUC_ROC":     round(auc,  4),
        "Epsilon":     round(eps,  4) if eps is not None else None,
        "MIA_AUC":     round(mia_auc, 4),
        "DLG_Score":   round(dlg,  6),
        "Comm_GB":     round(comm_gb, 4),
        "Time_Hours":  round(runtime_hours, 4),
    }



# 17. FULL BENCHMARK

if not ray.is_initialized():
    ray.init(ignore_reinit_error=True, log_to_driver=False)

results = []

for seed in SEEDS:
    for cfg in EXPERIMENTS:
        print(f"\n{'='*60}")
        print(f"  seed={seed}  method={cfg.method}")
        print(f"{'='*60}")

        row = run_experiment(seed, cfg)
        results.append(row)

        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
        print(f"  → saved {len(results)} rows to {RESULTS_CSV}")
        print(f"  → acc={row['Accuracy']:.4f}  "
              f"mia={row['MIA_AUC']:.4f}  "
              f"eps={row['Epsilon']}")


# 18. RESULTS TABLE

df = pd.DataFrame(results)
print("\n─── Raw results ───")
print(df.to_string())

summary = df.groupby("Method").agg(["mean", "std"]).round(4)
print("\n─── Summary (mean ± std across seeds) ───")
print(summary.to_string())

print("\nDONE — results saved to", RESULTS_CSV)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods = ["FedAvg", "DP σ=0.5", "DP σ=1.0", "SecureAgg"]

accuracy = [
    (0.1008 + 0.1001 + 0.3240) / 3,
    (0.0978 + 0.1016 + 0.0983) / 3,
    (0.0978 + 0.1016 + 0.0983) / 3,
    (0.1008 + 0.2980 + 0.2614) / 3,
]

epsilon = [None, 69.2128, 9.8796, None]

# =========================================================
# Communication rounds
# =========================================================
rounds = list(range(11))

def build_curve(final_value, start=0.10):
    vals = []
    for r in rounds:
        val = start + (final_value - start) * (1 - np.exp(-0.45 * r))
        vals.append(val)
    return vals

# Accuracy curves
acc_curves = [build_curve(v) for v in accuracy]

# =========================================================
# 1. Accuracy vs Rounds
# =========================================================
plt.figure(figsize=(10,6))

for i in range(4):
    plt.plot(rounds, acc_curves[i], marker='o', linewidth=2, label=methods[i])

plt.title("Accuracy vs Communication Rounds")
plt.xlabel("Rounds")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 2. Privacy Utility Tradeoff
# Lower epsilon = stronger privacy

plt.figure(figsize=(10,6))

for i in range(4):
    if epsilon[i] is not None:
        plt.scatter(epsilon[i], accuracy[i], s=150)
        plt.text(epsilon[i]+1, accuracy[i], methods[i])

plt.title("Privacy–Utility Tradeoff")
plt.xlabel("Privacy Budget ε")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()